In [1]:
# import libraries for reading data
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image
import cv2
import re
import torch
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torchvision import transforms, models
import ast
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights, efficientnet_v2_s, EfficientNet_V2_S_Weights
import torch.nn as nn
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from concurrent.futures import ProcessPoolExecutor
from PIL import ImageEnhance, Image
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from concurrent.futures import ThreadPoolExecutor
import torch.multiprocessing as mp
from torch.utils.data import WeightedRandomSampler
mp.set_sharing_strategy('file_system')

<jemalloc>: Unsupported system page size


### Einlesen der Daten und Übersicht über die Daten

In [2]:
# data paths
train1_images_path = "/datasets/multi-view-pig-posture-recognition/train1_images"
train2_images_path = "/datasets/multi-view-pig-posture-recognition/train2_images"
test_images_path = "/datasets/multi-view-pig-posture-recognition/test_images"

# csv path with row_id, image_id, width, height, bbox, class_id
train1_csv_path = "/datasets/multi-view-pig-posture-recognition/train1.csv"
train2_csv_path = "/datasets/multi-view-pig-posture-recognition/train2.csv"
test_csv_path = "/datasets/multi-view-pig-posture-recognition/test.csv"

# txt file path
pig_posture_txt = "/datasets/multi-view-pig-posture-recognition/pig_posture_classes.txt"

In [3]:
# read all files and show statistics and content of csv files, column names etc.
# read csv files
train1_df = pd.read_csv(train1_csv_path)
train2_df = pd.read_csv(train2_csv_path)
test_df = pd.read_csv(test_csv_path)

# show column names of csv files
print("\nTrain1 CSV Columns:")
print(train1_df.columns)
print("\nTrain2 CSV Columns:")
print(train2_df.columns)
print("\nTest CSV Columns:")
print(test_df.columns)

# show content of txt file
with open(pig_posture_txt, 'r') as f:
    pig_posture_content = f.read()

# show numbers of unique image_ids, row_ids in train1, train2 and test csv files
print("\nNumber of unique image_ids in Train1 CSV:", train1_df['image_id'].nunique())
print("Number of unique image_ids in Train2 CSV:", train2_df['image_id'].nunique())
print("Number of unique row_ids in Train1 CSV:", train1_df['row_id'].nunique())
print("Number of unique row_ids in Train2 CSV:", train2_df['row_id'].nunique())
print("\nPig Posture Classes:")
print(pig_posture_content)




Train1 CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id'], dtype='object')

Train2 CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id'], dtype='object')

Test CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox'], dtype='object')

Number of unique image_ids in Train1 CSV: 3090
Number of unique image_ids in Train2 CSV: 3150
Number of unique row_ids in Train1 CSV: 22934
Number of unique row_ids in Train2 CSV: 23450

Pig Posture Classes:
Lateral_lying_left
Lateral_lying_right
Sitting
Standing
Sternal_lying



In [4]:
train1_df.head()

,row_id,image_id,width,height,bbox,class_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3


In [5]:
train2_df.head()

,row_id,image_id,width,height,bbox,class_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3


### EDA 
Class Definitions:

0 — Lateral_lying_left

1 — Lateral_lying_right

2 — Sitting

3 — Standing

4 — Sternal_lying

### Klassen sind stark unausgewogen, insbesondere Sitting Class id = 2. Gegenmaßnahme ist notwendig, um die Minderheitsklasse nicht zu vernachlässigen.

In [6]:
# check if there are any missing values in train1 and train2 csv files
print("\nMissing values in Train1 CSV:")
print(train1_df.isnull().sum())
print("\nMissing values in Train2 CSV:")
print(train2_df.isnull().sum())


Missing values in Train1 CSV:
row_id      0
image_id    0
width       0
height      0
bbox        0
class_id    0
dtype: int64

Missing values in Train2 CSV:
row_id      0
image_id    0
width       0
height      0
bbox        0
class_id    0
dtype: int64


In [7]:
# check if unique values in "height" and "weight" columns in train1 and train2 csv files are the same
print("\nUnique values in 'height' column in Train1 CSV:")
print(train1_df['height'].unique())
print("\nUnique values in 'height' column in Train2 CSV:")
print(train2_df['height'].unique())
print("\nUnique values in 'width' column in Train1 CSV:")
print(train1_df['width'].unique())
print("\nUnique values in 'width' column in Train2 CSV:")
print(train2_df['width'].unique())


Unique values in 'height' column in Train1 CSV:
[1080  720 1520]

Unique values in 'height' column in Train2 CSV:
[1080  720 1520]

Unique values in 'width' column in Train1 CSV:
[1920 1280 2688]

Unique values in 'width' column in Train2 CSV:
[1920 1280 2688]


### Die Bilder liegen nur in drei Auflösungen vor: 1280 x 720, 1920 x 1080, 2688 x 1520. Vorverarbeitung ist konsistent planbar. 

### Fazit: Die EDA zeigt, dass die Klassen in den Trainingsdaten relativ ausgewogen verteilt sind, was für das Training eines Modells vorteilhaft ist. Es gibt keine fehlenden Werte in den CSV-Dateien, und die Bildgrößen sind konsistent. Die Analyse der Bildqualität anhand von Blur-Score und Helligkeit zeigt eine gewisse Variation. Im nächsten Schritt möchte ich die Kameras trennen und die Bilder entsprechend der Kamera analysieren, um mögliche Unterschiede in der Bildqualität oder den Aufnahmewinkeln zu identifizieren.

### Trennung der Kameras anhand der Bildnamen nur in Train1

In [8]:
def extract_pen_id(s: str):
    m = re.search(r'^(pen\d+)', s)
    return m.group(1) if m else None

def extract_camera_type(s: str):
    m = re.search(r'_(orb|tur)_', s)
    return m.group(1) if m else None

def extract_camera_number(s: str):
    m = re.search(r'cam(\d+)', s)
    return m.group(1) if m else None

# df1 = train1.csv als DataFrame; ersetze 'FILENAME_COL' durch deine Spalte (z. B. 'image', 'file_name', ...).
FILENAME_COL = "image_id"
train1_df["pen_id"]        = train1_df[FILENAME_COL].apply(extract_pen_id)
train1_df["camera_type"]   = train1_df[FILENAME_COL].apply(extract_camera_type)
train1_df["camera_number"] = train1_df[FILENAME_COL].apply(extract_camera_number)
train1_df["camera_view_id"]      = train1_df["pen_id"] + "_" + train1_df["camera_type"] + "_cam" + train1_df["camera_number"]


In [9]:
train1_df_distribution = train1_df['class_id'].value_counts().sort_index()
train2_df_distribution = train2_df['class_id'].value_counts().sort_index()

print("\nClass Distribution in Train1 CSV:")
print(train1_df_distribution)
print("\nClass Distribution in Train2 CSV:")
print(train2_df_distribution)


Class Distribution in Train1 CSV:
class_id
0    3053
1    3376
2     680
3    9617
4    6208
Name: count, dtype: int64

Class Distribution in Train2 CSV:
class_id
0    3083
1    3435
2     695
3    9928
4    6309
Name: count, dtype: int64


### Trennung der Kameras anhand der Bildnamen nur in Train2

In [10]:
train2_df["pen_id"]        = train2_df["image_id"].apply(extract_pen_id)
train2_df["camera_type"]   = train2_df["image_id"].apply(extract_camera_type)
train2_df["camera_number"] = train2_df["image_id"].apply(extract_camera_number)
train2_df["camera_view_id"]      = train2_df["pen_id"] + "_" + train2_df["camera_type"] + "_cam" + train2_df["camera_number"]


In [11]:
# transforms for data augmentation and data preprocessing
img_size = (224, 224)
normalize_mean = [0.485, 0.456, 0.406]
normalize_std = [0.229, 0.224, 0.225]


train_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),           # NEU: Simuliert andere Kamerawinkel
    transforms.RandomRotation(15),                    # Erhöht von 10 auf 15
    transforms.RandomAffine(degrees=0, scale=(0.8, 1.2), shear=10),  # NEU: Perspektive
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15),  # Stärker
    transforms.RandomGrayscale(p=0.1),               # NEU: Robustheit gegen Farbverschiebung
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.2),                  # NEU: Simuliert Verdeckungen
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])

val_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])



### Anmerkung 04.03.26 : Neue train_transforms-Inhalte und Ideen.

In [12]:
class PigCropDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, preload=True):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.images = []
        self.labels = []
        
        if preload:
            print(f"Lade {len(self.df)} Bilder mit 120 Cores in den RAM...")
            
            def load_single_image(idx):
                r = self.df.iloc[idx]
                p = os.path.join(self.image_dir, r["image_id"])
                # Direktes Laden und Zuschneiden
                img = Image.open(p).convert("RGB")
                bbox = ast.literal_eval(r["bbox"]) if isinstance(r["bbox"], str) else r["bbox"]
                x, y, w, h = bbox
                # Resize hier spart massiv RAM und CPU-Zeit beim Training
                crop = img.crop((x, y, x + w, y + h)).resize((224, 224))
                return crop, int(r["class_id"])

            # Wir nutzen 120 der 160 Cores, um das System nicht komplett zu blockieren
            with ThreadPoolExecutor(max_workers=120) as executor:
                results = list(tqdm(executor.map(load_single_image, range(len(self.df))), total=len(self.df)))
            
            self.images, self.labels = zip(*results)
            print("Preloading abgeschlossen. Daten liegen nun im RAM.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Kein Disk-Zugriff mehr! Nur noch RAM-Zugriff.
        img = self.images[idx]
        label = self.labels[idx]
        
        if self.transform:
            img = self.transform(img)
            
        return img, torch.tensor(label, dtype=torch.long)


In [13]:
sampleDS = PigCropDataset(train2_df, train2_images_path, transform=train_transforms)
loader = DataLoader(sampleDS, batch_size=9, shuffle=True)
for x, y in loader:
    print("Batch Image Shape:", x.shape)  # erwartet: [9, 3, 224, 224]
    print("Batch Label Shape:", y.shape)  # erwartet: [9]
    print("First Label:", y[0].item())    # 0..4
    break


Lade 23450 Bilder mit 120 Cores in den RAM...


100%|██████████| 23450/23450 [00:14<00:00, 1618.41it/s] 


Preloading abgeschlossen. Daten liegen nun im RAM.
Batch Image Shape: torch.Size([9, 3, 224, 224])
Batch Label Shape: torch.Size([9])
First Label: 3


In [14]:
class PigPostureCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(PigPostureCNN, self).__init__()
        
        weights = EfficientNet_V2_S_Weights.DEFAULT
        original_model = efficientnet_v2_s(weights=weights)
        
        
        self.backbone = original_model.features
        self.pool = nn.AdaptiveAvgPool2d(1) # Global Average Pooling
        
        # EfficientNetV2-S hat am Ende der Features 1280 Kanäle
        in_features = 1280 
        
        self.label_classifier = nn.Sequential(
            nn.Linear(in_features, 512), # 512 ist bei EfficientNet besser als 256
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)       # Das hier hat in deinem Code gefehlt
        x = torch.flatten(x, 1)
        label_preds = self.label_classifier(x)
        return label_preds
        
# Instanziierung für alle verfügbaren GPUs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PigPostureCNN(num_classes=5)

# Multi-GPU Parallelisierung aktivieren
if torch.cuda.device_count() > 1:
    print(f"Nutze {torch.cuda.device_count()} GPUs für das Training!")
    model = nn.DataParallel(model)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /home/jovyan/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


  0%|          | 0.00/82.7M [00:00<?, ?B/s]

Nutze 4 GPUs für das Training!


### Einfügen einer Metrik, um mit der Klassenverteilung besser ausheben zu können. Den unterrepräsentierten Klassen eine höhere Gewichtung geben, um die Ungleichheit der Klassenverteilung zu adressieren. 

In [15]:
class Learner:
    def __init__(self, model, train_dl, val_dl, device=None, class_weights=None):
        self.model = model
        self.train_dl = train_dl
        self.val_dl = val_dl
        self.device = device
        
        self.model = self.model.to(self.device)
        
        # === Gewichtete Loss-Funktion MIT Label Smoothing ===
        if class_weights is not None:
            self.loss_fn_classifier = nn.CrossEntropyLoss(
                weight=class_weights.to(self.device),
                label_smoothing=0.1  # NEU: Verhindert Overconfidence
            )
            print(f"Klassen-Gewichte: {class_weights}")
        else:
            self.loss_fn_classifier = nn.CrossEntropyLoss(label_smoothing=0.1)
        
        self.best_acc = 0
        self.best_model_state = None  # NEU: Speichert das beste Modell
        self.scaler = torch.cuda.amp.GradScaler()
        self.freeze()
        
    def freeze(self):
        actual_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for param in actual_model.backbone.parameters():
            param.requires_grad = False
        
    def unfreeze(self):
        actual_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for param in actual_model.backbone.parameters():
            param.requires_grad = True

    @torch.no_grad()
    def validate(self):
        """Berechnet Val-Loss und Val-Accuracy nach jeder Epoche."""
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        for xb, yb in self.val_dl:
            xb, yb = xb.to(self.device, non_blocking=True), yb.to(self.device, non_blocking=True)
            with torch.cuda.amp.autocast():
                preds = self.model(xb)
                loss = self.loss_fn_classifier(preds, yb)
            total_loss += loss.item() * xb.size(0)
            correct += (preds.argmax(1) == yb).sum().item()
            total += xb.size(0)
        
        self.model.train()
        return total_loss / total, correct / total

    def fit(self, epochs, lr=1e-3, early_stopping_patience=5):
        self.optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.model.parameters()), 
            lr=lr
        )
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer, max_lr=lr*10, total_steps=epochs*len(self.train_dl)
        )
        
        # Early Stopping Variablen
        patience_counter = 0
        best_val_loss = float('inf')
        
        for epoch in range(epochs):
            self.model.train()
            train_loss_sum = 0
            train_correct = 0
            train_total = 0
            
            for xb, yb in tqdm(self.train_dl, desc=f"Epoch {epoch+1}/{epochs}"):
                xb, yb = xb.to(self.device, non_blocking=True), yb.to(self.device, non_blocking=True)
                self.optimizer.zero_grad()
                
                with torch.cuda.amp.autocast():
                    preds = self.model(xb)
                    loss = self.loss_fn_classifier(preds, yb)
                
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step()
                
                # Training-Metriken sammeln
                train_loss_sum += loss.item() * xb.size(0)
                train_correct += (preds.argmax(1) == yb).sum().item()
                train_total += xb.size(0)
            
            # === Validation nach jeder Epoche ===
            val_loss, val_acc = self.validate()
            train_loss = train_loss_sum / train_total
            train_acc = train_correct / train_total
            
            print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
            
            # === Early Stopping Check ===
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                # Bestes Modell speichern
                self.best_model_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✅ Neues bestes Modell gespeichert (Val Loss: {val_loss:.4f})")
            else:
                patience_counter += 1
                print(f"  ⚠️ Keine Verbesserung seit {patience_counter}/{early_stopping_patience} Epochen")
                
            if patience_counter >= early_stopping_patience:
                print(f"  🛑 Early Stopping nach Epoche {epoch+1}!")
                break
        
        # Am Ende das beste Modell laden
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            print("✅ Bestes Modell wiederhergestellt.")

### Optimierte Learner-Klasse (Änderungen gegenüber vorher):
1. **Label Smoothing (label_smoothing=0.1)**: Vorher hat das Modell gelernt, 100% sicher 
bei seinen Vorhersagen zu sein (z.B. "Das ist zu 99.9% Standing"). Das führt zu Overfitting.
Label Smoothing sagt dem Modell: "Sei dir nie ganz sicher" → bessere Generalisierung.

2. **Validation nach jeder Epoche (validate-Methode)**: Vorher wurde nur trainiert, aber nie 
geprüft, wie gut das Modell auf ungesehenen Daten ist. Man flog also blind. Jetzt sieht man 
nach jeder Epoche Train Loss/Acc UND Val Loss/Acc → man erkennt sofort, wann Overfitting beginnt.

3. **Early Stopping (patience=5)**: Vorher wurde stur eine fixe Anzahl Epochen trainiert, 
auch wenn das Modell längst nicht mehr besser wurde (oder sogar schlechter). Jetzt stoppt 
das Training automatisch, wenn der Val Loss 5 Epochen lang nicht sinkt.

4. **Bestes Modell speichern/wiederherstellen**: Vorher wurde am Ende einfach der letzte 
Zustand genommen – der ist oft schlechter als der beste Zwischenstand. Jetzt wird das 
Modell mit dem niedrigsten Val Loss gespeichert und am Ende wiederhergestellt.

5. **class_weights als Parameter**: Vorher war class_weights eine globale Variable im 
__init__ – das ist fehleranfällig. Jetzt wird es sauber als Parameter übergeben.

In [16]:
# GroupShuffleSplit nach image_id: Alle Schweine aus einem Bild bleiben zusammen,
# aber verschiedene Bilder der GLEICHEN Kamera können in Train UND Val sein.
# Das verhindert Data Leakage auf Bild-Ebene, gibt aber trotzdem einen sauberen 80/20 Split.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(gss.split(train2_df, groups=train2_df['image_id']))

train_data = train2_df.iloc[train_idx].copy()
val_data = train2_df.iloc[val_idx].copy()

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Train Kameras: {train_data['camera_view_id'].nunique()}")
print(f"Val Kameras: {val_data['camera_view_id'].nunique()}")

Training samples: 18802
Validation samples: 4648
Train Kameras: 8
Val Kameras: 8




So sehen wir, welcher Fold eine sinnvolle Train/Val-Verteilung hat (idealerweise ~80/20). Hier ist die Fold-Übersicht:

Fold 1: Train=13737, Val=9713,  Val-Kameras: ['pen2_tur_cam1']           → 59/41 Split ❌
Fold 2: Train=15256, Val=8194,  Val-Kameras: ['pen1_tur_cam2']           → 65/35 Split ❌
Fold 3: Train=20633, Val=2817,  Val-Kameras: ['pen2_orb_cam1']           → 88/12 Split ✅
Fold 4: Train=21962, Val=1488,  Val-Kameras: ['pen1_orb_cam2']           → 94/6  Split ⚠️
Fold 5: Train=22212, Val=1238,  Val-Kameras: 4 Kameras gemischt          → 95/5  Split ⚠️

Das Problem wird jetzt klar sichtbar

Die Kameras haben extrem unterschiedliche Anzahlen an Bildern. Die tur-Kameras (Fold 1 & 2) haben jeweils tausende Bilder, während die orb-Kameras deutlich weniger haben.
Welcher Fold ist der beste?

Fold 3 ist der sinnvollste Kompromiss:

    88/12 Split – nahe an der üblichen 80/20 Regel
    Trainiert auf 20.633 Bildern (fast alle Daten)
    Val-Set mit 2.817 Bildern – genug für verlässliche Metriken
    Nur eine Kamera im Val → echte Generalisierungsprüfung

Aber: GroupKFold ist hier grundsätzlich problematisch

Das Problem ist, dass du nur 8 camera_view_ids hast, die aber sehr ungleich groß sind. Kein Fold ergibt einen sauberen 80/20 Split.

Mein Vorschlag für NB10: Geh zurück zu GroupShuffleSplit, aber mit einer wichtigen Verbesserung – gruppiere nach image_id statt nach camera_view_id:



In [17]:
# === Klassen-Gewichte berechnen (inverse Häufigkeit) ===
class_counts = train_data['class_id'].value_counts().sort_index()
total = len(train_data)
class_weights = torch.tensor(
    total / (len(class_counts) * class_counts.values), 
    dtype=torch.float32
)
print(f"Klassen-Gewichte: {class_weights}")

# KEIN WeightedRandomSampler mehr! Die Gewichtung passiert nur im Loss.

Klassen-Gewichte: tensor([1.5292, 1.3709, 6.5856, 0.4721, 0.7426])


### Klassen-Balancierung: Nur noch über Weighted Loss

 VORHER: WeightedRandomSampler + Weighted Loss gleichzeitig
   → Doppelte Übergewichtung der seltenen Klasse "Sitting":
     1. Sitting wird häufiger gezogen (Sampler)
     2. UND Sitting-Fehler werden stärker bestraft (Loss)
   → Das kann dazu führen, dass das Modell zu oft Sitting vorhersagt.

 JETZT: Nur Weighted Loss (kein Sampler)
   → Die Klassen-Gewichte sorgen dafür, dass Fehler bei seltenen Klassen 
     stärker bestraft werden, aber die natürliche Verteilung der Daten bleibt erhalten.
   → Sitting hat Gewicht 13.27, Standing nur 0.52 → ein Sitting-Fehler 
     "kostet" das Modell ~25x mehr als ein Standing-Fehler.

 ### Training Phase 1 (Änderungen):

 1. **shuffle=True statt sampler=sampler**: Da wir den WeightedRandomSampler 
    entfernt haben, nutzen wir normales Shuffling.

 2. **8 statt 15 Epochen**: Der Classifier-Kopf konvergiert schnell auf den 
    fixen Backbone-Features. 15 Epochen waren zu viel → der Kopf hat sich 
    an die fixen Features "überangepasst". 8 Epochen + Early Stopping reicht.

In [18]:
# --- VORBEREITUNG ---
# Nutze hier die Version von PigCropDataset, in die wir das Caching (RAM-Speicher) 
# eingebaut haben, damit es ab Epoche 2 extrem schnell geht.
train_ds = PigCropDataset(train_data, train2_images_path, transform=train_transforms)
val_ds = PigCropDataset(val_data, train2_images_path, transform=val_transforms)

# DataLoaders - Optimierung: pin_memory auch für Validierung nutzen
# Erhöhe die Batch Size auf 4096 (1024 Bilder pro GPU)
# Sollte das ein OOM (Out of Memory) geben, geh auf 3072 zurück.
# Batch-Size pro Schritt (wird auf 4 GPUs verteilt -> 256 pro Karte)
batch_size = 1024 

train_dl = DataLoader(
    train_ds, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=8,               # 8 Worker sind stabil und schnell genug für RAM-Daten
    pin_memory=True,             # Schaufelt Daten schneller in den VRAM
    prefetch_factor=2, 
    persistent_workers=True
)

val_dl = DataLoader(
    val_ds, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True,
    persistent_workers=True
)
# 1. Modell & Learner vorbereiten (wie bisher)

learner = Learner(model, train_dl, val_dl, device=device, class_weights=class_weights)

# --- PHASE 1: Classifier-Training (Frozen Backbone) ---
# Wir geben dem Kopf 15 Epochen, um sich an die Features zu gewöhnen
print("Starte Phase 1: Classifier Fine-Tuning...")
learner.freeze()
learner.fit(epochs=8, lr=1e-3, early_stopping_patience=5)

Lade 18802 Bilder mit 120 Cores in den RAM...


100%|██████████| 18802/18802 [00:01<00:00, 17548.87it/s]


Preloading abgeschlossen. Daten liegen nun im RAM.
Lade 4648 Bilder mit 120 Cores in den RAM...


100%|██████████| 4648/4648 [00:06<00:00, 734.78it/s]  


Preloading abgeschlossen. Daten liegen nun im RAM.
Klassen-Gewichte: tensor([1.5292, 1.3709, 6.5856, 0.4721, 0.7426])
Starte Phase 1: Classifier Fine-Tuning...


...

  Train Loss: 1.3907 | Train Acc: 0.5226 | Val Loss: 1.2830 | Val Acc: 0.6119
  ✅ Neues bestes Modell gespeichert (Val Loss: 1.2830)


Epoch 7/8: 100%|██████████| 19/19 [00:29<00:00,  1.58s/it]


  Train Loss: 1.3677 | Train Acc: 0.5494 | Val Loss: 1.2698 | Val Acc: 0.6123
  ✅ Neues bestes Modell gespeichert (Val Loss: 1.2698)


Epoch 8/8: 100%|██████████| 19/19 [00:32<00:00,  1.74s/it]


  Train Loss: 1.3609 | Train Acc: 0.5465 | Val Loss: 1.2704 | Val Acc: 0.6022
  ⚠️ Keine Verbesserung seit 1/5 Epochen
✅ Bestes Modell wiederhergestellt.


### Training Phase 2 (Änderungen):

 1. **20 statt 35 Epochen**: Weniger Epochen = weniger Overfitting-Risiko.
    Durch Early Stopping wird ohnehin früher gestoppt, wenn nötig.

 2. **lr=1e-4 statt 5e-5**: Etwas höhere Lernrate, damit das Modell in den 
    weniger Epochen noch genug lernen kann. 5e-5 war zu konservativ.

In [19]:
import gc
torch.cuda.empty_cache()
gc.collect()

batch_size_phase2 = 512

train_dl = DataLoader(train_ds, batch_size=batch_size_phase2, shuffle=True,
                      num_workers=16, pin_memory=True, persistent_workers=True)
val_dl = DataLoader(val_ds, batch_size=batch_size_phase2, shuffle=False,
                    num_workers=8, pin_memory=True, persistent_workers=True)

learner.train_dl = train_dl
learner.val_dl = val_dl

# --- PHASE 2: Alles offen, niedrige LR ---
print("Starte Phase 2: Full Model Fine-Tuning...")
learner.unfreeze()
learner.fit(epochs=20, lr=1e-4, early_stopping_patience=5)

Starte Phase 2: Full Model Fine-Tuning...


Epoch 1/20: 100%|██████████| 37/37 [00:36<00:00,  1.02it/s]


  Train Loss: 1.3131 | Train Acc: 0.5948 | Val Loss: 1.1813 | Val Acc: 0.7083
  ✅ Neues bestes Modell gespeichert (Val Loss: 1.1813)


Epoch 2/20: 100%|██████████| 37/37 [00:29<00:00,  1.24it/s]


  Train Loss: 1.1742 | Train Acc: 0.6815 | Val Loss: 1.0815 | Val Acc: 0.7261
  ✅ Neues bestes Modell gespeichert (Val Loss: 1.0815)


...

  Train Loss: 0.6965 | Train Acc: 0.9842 | Val Loss: 0.7755 | Val Acc: 0.9606
  ⚠️ Keine Verbesserung seit 1/5 Epochen


Epoch 19/20: 100%|██████████| 37/37 [00:30<00:00,  1.22it/s]


  Train Loss: 0.6955 | Train Acc: 0.9849 | Val Loss: 0.7733 | Val Acc: 0.9621
  ⚠️ Keine Verbesserung seit 2/5 Epochen


Epoch 20/20: 100%|██████████| 37/37 [00:31<00:00,  1.16it/s]


  Train Loss: 0.6963 | Train Acc: 0.9845 | Val Loss: 0.7732 | Val Acc: 0.9630
  ⚠️ Keine Verbesserung seit 3/5 Epochen
✅ Bestes Modell wiederhergestellt.


In [20]:
# 1. Test-Daten vorbereiten
test_df_copy = test_df.copy()
test_df_copy['class_id'] = 0 

# Sicherheitshalber eine moderate Batch-Size wählen (z.B. 512 oder 256)
batch_size_inference = 512 

test_ds = PigCropDataset(test_df_copy, test_images_path, transform=val_transforms)
test_dl = DataLoader(test_ds, batch_size=batch_size_inference, shuffle=False, 
                    num_workers=8, pin_memory=True)

# 2. Modell in den Vorhersage-Modus schalten
model.eval()
all_predictions = []

print(f"Erstelle Vorhersagen auf {device}...")

with torch.no_grad():
    # NUTZE AMP: Das spart VRAM und beschleunigt die Vorhersage auf V100 massiv
    with torch.cuda.amp.autocast():
        for xb, _ in tqdm(test_dl):
            # non_blocking=True für schnelleren Datentransfer
            xb = xb.to(device, non_blocking=True) 
            
            outputs = model(xb)
            
            # Klasse mit dem höchsten Wert
            _, preds = torch.max(outputs, 1)
            
            all_predictions.extend(preds.cpu().numpy())

# 3. Die finale CSV-Datei erstellen
submission = pd.DataFrame({
    'row_id': test_df['row_id'],
    'class_id': all_predictions
})

# Speichern
submission_name = '6th_submission.csv'
submission.to_csv(submission_name, index=False)

print(f"Erfolgreich! Die Datei '{submission_name}' mit {len(submission)} Zeilen wurde erstellt.")

Lade 11708 Bilder mit 120 Cores in den RAM...


100%|██████████| 11708/11708 [00:04<00:00, 2908.07it/s]


Preloading abgeschlossen. Daten liegen nun im RAM.
Erstelle Vorhersagen auf cuda...


100%|██████████| 23/23 [00:09<00:00,  2.37it/s]

Erfolgreich! Die Datei '6th_submission.csv' mit 11708 Zeilen wurde erstellt.
